In [ ]:
"""
exp12 Step 02: Continuous Replay On The Validation Span (PROTOCOL.md §5-§9)

One continuous path over the 44 validation quarters of the exp02-exp07 schedule (2015-04-17 -> 2026-04-15); the position
carries across quarters and nothing forces a buy-back. SPY bars and every VIX load are cut at 2026-04-15.
    0. REPRODUCTION (before any exp12 number): the unmodified E1 (exp04 x = 0, n = 1) and E2 (exp07 k = 4) over the 44
       periods (+108.03% with 35 exits, +115.60% with 12 exits), buy-and-hold over the full span (+235.39%) and over the
       prior-only span from 2015-07-17 (+229.74%). Any difference stops the notebook.
    1. CANDIDATE PATHS (after the fact, optimistic for the best): the 6 candidates, each replayed over the whole span; its
       return in each period minus buy-and-hold's = its period score (6 x 44 = 264 validation trials).
    2. SELECTION: mean period score over the last 4 periods; ties by the fixed rank of config.TIE_ORDER_DICT (E2M3 first).
    3. PRIOR-ONLY PATH (the honest estimate): the candidate selected at period f governs period f + 1; one continuous
       switching path with buy-and-hold, 20 random-exit and 20 random-re-entry runs, the log-excess intervals, the cost
       sensitivity, the episode scorecard (re-entry reason, VIX at exit and buy-back) and the concentration reports.
    4. INFORMATION TEST: the path must also beat the UNMODIFIED prior-only path over {E1, E2} (same selection) and the
       median of both random families.

STOP (PROTOCOL.md §8) if the prior-only path does not beat buy-and-hold: the VIX line closes definitively. A pass is only
a "candidate result pending review". RUN_MODE: "validation_only" only (the untouched window 2026-05-14 -> 2026-08-13 is
never read). Prerequisite: the raw data only. Outputs: store04_experiments/exp12_vix_fear_reentry/step02_continuous_replay_data/.
"""
None

In [ ]:
# IMPORT GENERAL PACKAGES
import pandas as pd
import numpy as np
import time
# IMPORT PLOTLY
import plotly.graph_objects as go
# IMPORT THE SHARED CONFIGURATION, THE EXPERIMENT CONFIGURATION AND THE PATHS
from so import config, paths
from experiments.exp12_vix_fear_reentry import config as exp_config
# IMPORT DATA LOADING, EXECUTION, DAILY FEATURE AND VIX FEATURE FUNCTIONS
from so.core.raw_data import get_complete_ohlcv_pdf
from so.core.trade_execution import get_ohlcv_array_dict
from so.features.daily_features import get_daily_feature_pdf
from so.features.vix_features import get_vix_daily_feature_dict
# IMPORT LOCAL FILE MANAGEMENT FUNCTIONS
from so.core.local_file_management import write_csv_file_to_path
# IMPORT THE CONTINUOUS REPLAY AND WALK-FORWARD FUNCTIONS
from so.core.continuous_replay import get_replay_period_pdf
from so.core.replay_walk_forward import run_candidate_replay_dict, run_prior_only_replay_dict
# IMPORT THE EXPERIMENT FUNCTIONS
from experiments.exp12_vix_fear_reentry.rules import get_schedule_tuple, check_schedule_dict, get_rule_candidate_list, get_unmodified_candidate_list, get_rule_builder_func, \
    get_vix_scorecard_pdf, get_concentration_dict, get_yearly_log_excess_pdf
# IMPORT TRIAL LOG FUNCTIONS
from so.core.trial_log import get_config_hash_str, log_trial_pdf, log_trial_dict, read_trial_log_pdf, get_trial_count_pdf, print_previous_trial_warning

In [ ]:
# DEFINE THE RUN MODE: ONLY "validation_only" EXISTS FOR THIS EXPERIMENT (NO TEST WINDOW IS EVALUATED)
RUN_MODE_STR = "validation_only"
assert RUN_MODE_STR == "validation_only"
# CHECK THE REAL SCHEDULE AND THE REPRODUCTION ONLY OUTSIDE THE SYNTHETIC SMOKE RUN (scripts/smoke_notebook.py)
CHECK_REAL_DATA_BOOL = __name__ != "__smoke__"
# DEFINE THE CANDIDATE KEYS AND THE TIE-BREAKS OF THE SELECTION (PROTOCOL.md §5: THE FIXED RANK, E2M3 FIRST)
KEY_COL_STR_LIST = ["exit_rule", "modification"]
TIE_BREAK_LIST = [("tie_order", True)]
# DISPLAY THE SETTINGS
print(f"Run Mode:\t{RUN_MODE_STR}\tCandidates:\t{[c['candidate_name'] for c in get_rule_candidate_list()]}\tTie order:\t{exp_config.TIE_ORDER_DICT}\tVIX timing:\t{exp_config.VIX_VARIANT_STR}")

In [ ]:
# DISPLAY THE EXPERIMENT, THE PROTOCOL VERSION AND THE CONFIGURATION HASH (SHARED + EXPERIMENT CONFIG)
print(f"Experiment:\t{exp_config.EXPERIMENT_NAME}\tProtocol:\t{exp_config.PROTOCOL_VERSION}\tConfig Hash:\t{get_config_hash_str(exp_config)}")
# DEFINE THE OUTPUT FOLDER OF THIS STEP
output_path_str = paths.get_experiment_data_path_str(exp_config.EXPERIMENT_NAME, "step02_continuous_replay_data")
print(f"Output folder:\t{output_path_str}")
# CALL FUNCTION TO WARN ABOUT EARLIER ENTRIES OF THIS STAGE (A RERUN ADDS TRIALS)
previous_trial_count = print_previous_trial_warning(exp_config, "validation")

In [ ]:
# CALL FUNCTION TO READ THE MINUTE BARS AND BUILD THE SCHEDULE FROM THE FULL SESSION CALENDAR (SAME 44 QUARTERS AS exp04 STEP 02)
complete_ohlcv_pdf = get_complete_ohlcv_pdf()
fold_pdf, _ = get_schedule_tuple(sorted(complete_ohlcv_pdf["date"].unique()))
# DEFINE THE LAST VALIDATION SESSION AND CUT THE BARS THERE (NO LATER BAR IS GIVEN TO THE SIMULATOR OR THE FEATURES)
replay_end_date = pd.to_datetime(fold_pdf["valid_end"].iloc[-1]).date()
assert replay_end_date < pd.Timestamp(exp_config.UNTOUCHED_START_DATE_STR).date()
replay_ohlcv_pdf = complete_ohlcv_pdf[complete_ohlcv_pdf["date"] <= replay_end_date].reset_index(drop=True)
del complete_ohlcv_pdf
# CALL FUNCTION TO CONVERT THE CUT DATA TO ARRAYS, BUILD THE DAILY TABLE AND THE REPLAY PERIODS
ohlcv_array_dict = get_ohlcv_array_dict(replay_ohlcv_pdf)
daily_pdf = get_daily_feature_pdf(ohlcv_array_dict)
period_pdf = get_replay_period_pdf(fold_pdf, ohlcv_array_dict["session_date_list"])
# CHECK THE SCHEDULE (44 PERIODS, 2015-04-17 -> 2026-04-15, LATEST TEST START 2026-05-14 NEVER READ)
if CHECK_REAL_DATA_BOOL:
    print(f"Schedule:\t{check_schedule_dict(fold_pdf, period_pdf)}")
# CALL FUNCTION TO LOAD THE VIX AND VIX3M DAILY BARS WITH THE SAME CUTOFF AND BUILD THE _prev FEATURES
vix_cutoff_date_str = str(replay_end_date)
vix_dict = get_vix_daily_feature_dict(daily_pdf, vix_cutoff_date_str, [exp_config.VIX_VARIANT_STR])
vix_feature_pdf = vix_dict["feature_pdf"]
assert all(last_date <= replay_end_date for last_date in vix_dict["last_date_dict"].values())
# SAVE THE SCHEDULE AND THE PERIODS
write_csv_file_to_path(fold_pdf, f"{output_path_str}fold_schedule_data.csv", "W", alert_in=False)
write_csv_file_to_path(period_pdf, f"{output_path_str}replay_period_data.csv", "W", alert_in=False)
# DISPLAY THE LAST DATES LOADED
print(f"Last SPY date loaded:\t{replay_ohlcv_pdf['date'].max()}\t({len(daily_pdf):,} sessions from {replay_ohlcv_pdf['date'].min()})")
for key_str, last_date in vix_dict["last_date_dict"].items():
    print(f"Last date loaded:\t{key_str}\t{last_date}")
print(f"Folds:\t{len(fold_pdf)}\tPeriods:\t{len(period_pdf)}\t{period_pdf['period_start'].iloc[0]} -> {period_pdf['period_end'].iloc[-1]}\t({int(period_pdf['session_count'].sum()):,} sessions)")
# DISPLAY THE COVERAGE OF THE THREE FEATURES ON THE REPLAY SPAN
span_feature_pdf = vix_feature_pdf[(vix_feature_pdf["date"] >= period_pdf["period_start"].iloc[0]) & (vix_feature_pdf["date"] <= period_pdf["period_end"].iloc[-1])]
print("Missing values on the replay span:", {col: int(span_feature_pdf[col].isna().sum()) for col in [exp_config.VIX_LEVEL_COL_STR, exp_config.VIX_TERM_COL_STR, exp_config.VIX_PCT250_COL_STR]})

In [ ]:
"""
Reproduction (before any exp12 number): the unmodified exits and buy-and-hold, and the unmodified prior-only path (the
baseline of the information test, computed, not logged).
"""
# DEFINE THE RULE BUILDER (SHARED BY THE UNMODIFIED EXITS AND THE CANDIDATES)
rule_builder_func = get_rule_builder_func(daily_pdf, vix_feature_pdf)
# CALL FUNCTION TO REPLAY THE UNMODIFIED E1 AND E2 OVER THE 44 PERIODS
start_time = time.time()
unmodified_list = get_unmodified_candidate_list()
unmodified_replay_dict = run_candidate_replay_dict(ohlcv_array_dict, period_pdf, unmodified_list, rule_builder_func, exp_config.UNTOUCHED_START_DATE_STR,
                                                   exp_config.MAX_CASH_SESSIONS, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT, alert_in=False)
unmodified_summary_pdf = unmodified_replay_dict["candidate_summary_pdf"]
# CALL FUNCTION TO RUN THE UNMODIFIED PRIOR-ONLY PATH (SAME SELECTION, E2 FIRST ON TIES) WITH ITS BASELINES
unmodified_prior_dict = run_prior_only_replay_dict(ohlcv_array_dict, period_pdf, unmodified_replay_dict["candidate_period_pdf"], unmodified_list, rule_builder_func,
                                                   KEY_COL_STR_LIST, TIE_BREAK_LIST, exp_config.SELECTION_POOLED_QUARTER_COUNT, exp_config.UNTOUCHED_START_DATE_STR,
                                                   exp_config.MAX_CASH_SESSIONS, exp_config.RANDOM_RUN_COUNT, exp_config.RANDOM_EXIT_SEED_OFFSET, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
unmodified_summary_dict = unmodified_prior_dict["summary_dict"]
print(f"Unmodified exits replayed in {time.time() - start_time:.0f} s")
# COLLECT THE REPRODUCTION NUMBERS
e1_row = unmodified_summary_pdf[unmodified_summary_pdf["exit_rule"] == "E1"].iloc[0]
e2_row = unmodified_summary_pdf[unmodified_summary_pdf["exit_rule"] == "E2"].iloc[0]
reproduction_dict = {"e1_total_pct": round(100 * e1_row["strategy_total_return"], 2), "e1_exit_count": int(e1_row["exit_count"]),
                     "e2_total_pct": round(100 * e2_row["strategy_total_return"], 2), "e2_exit_count": int(e2_row["exit_count"]),
                     "buy_hold_full_pct": round(100 * unmodified_replay_dict["buy_hold_dict"]["metric_dict"]["total_return"], 2),
                     "buy_hold_prior_only_pct": round(100 * unmodified_summary_dict["buy_hold_total_return"], 2),
                     "prior_only_first_date": str(unmodified_summary_dict["first_date"])}
print(f"Reproduction:\t{reproduction_dict}")
# ASSERT THE REPRODUCTION (A DIFFERENCE STOPS THE NOTEBOOK: RECORD IT AND PARK IT)
if CHECK_REAL_DATA_BOOL:
    assert abs(reproduction_dict["e1_total_pct"] - exp_config.REPRODUCTION_E1_TOTAL_PCT) < 1e-9 and reproduction_dict["e1_exit_count"] == exp_config.REPRODUCTION_E1_EXIT_COUNT, "❌ E1 differs"
    assert abs(reproduction_dict["e2_total_pct"] - exp_config.REPRODUCTION_E2_TOTAL_PCT) < 1e-9 and reproduction_dict["e2_exit_count"] == exp_config.REPRODUCTION_E2_EXIT_COUNT, "❌ E2 differs"
    assert abs(reproduction_dict["buy_hold_full_pct"] - exp_config.REPRODUCTION_BUY_HOLD_FULL_PCT) < 1e-9, "❌ Buy-and-hold (full span) differs"
    assert abs(reproduction_dict["buy_hold_prior_only_pct"] - exp_config.REPRODUCTION_BUY_HOLD_PRIOR_ONLY_PCT) < 1e-9, "❌ Buy-and-hold (prior-only span) differs"
    assert reproduction_dict["prior_only_first_date"] == exp_config.EXPECTED_PRIOR_ONLY_START_STR, "❌ Prior-only span differs"
    print("Reproduction asserts:\tPASSED")
# DISPLAY THE UNMODIFIED PRIOR-ONLY PATH (THE BASELINE OF THE INFORMATION TEST)
print(f"Unmodified prior-only path:\t{unmodified_summary_dict['strategy_total_return']:.2%} vs buy & hold {unmodified_summary_dict['buy_hold_total_return']:.2%}"
      f"\tin market {unmodified_summary_dict['in_market_pct']:.1%}\texits {unmodified_prior_dict['simulation_dict']['metric_dict']['signal_exit_count']}"
      f"\tlog excess {unmodified_summary_dict['log_excess_annualized_log_excess']:.4f} [{unmodified_summary_dict['log_excess_ci_low']:.4f}, {unmodified_summary_dict['log_excess_ci_high']:.4f}]")
# SAVE THE BASELINE
write_csv_file_to_path(unmodified_summary_pdf, f"{output_path_str}unmodified_candidate_summary_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame([{key: value for key, value in unmodified_summary_dict.items() if key != "baselines_beaten"}]), f"{output_path_str}unmodified_prior_only_summary_data.csv", "W", alert_in=False)
write_csv_file_to_path(unmodified_prior_dict["selection_pdf"], f"{output_path_str}unmodified_selection_data.csv", "W", alert_in=False)
# PREVIEW DATAFRAME
unmodified_summary_pdf[["candidate_name", "strategy_total_return", "buy_hold_total_return", "in_market_pct", "exit_count", "bought_back_lower_pct"]].round(4)

In [ ]:
"""
Candidate paths (AFTER THE FACT: the best of 6 chosen with hindsight is optimistic).
"""
# CHECK THE TRIAL BUDGET BEFORE LOGGING (EVERY STAGE OF THIS EXPERIMENT; THIS STEP ADDS 264 + 1)
candidate_list = get_rule_candidate_list()
trial_log_pdf = read_trial_log_pdf()
experiment_trial_count = int((trial_log_pdf["experiment"] == exp_config.EXPERIMENT_NAME).sum()) if len(trial_log_pdf) else 0
assert experiment_trial_count + len(candidate_list) * len(period_pdf) + 1 <= exp_config.TRIAL_BUDGET, "❌ Trial budget exceeded"
# CALL FUNCTION TO REPLAY EVERY CANDIDATE AND SCORE IT PER PERIOD
start_time = time.time()
candidate_replay_dict = run_candidate_replay_dict(ohlcv_array_dict, period_pdf, candidate_list, rule_builder_func, exp_config.UNTOUCHED_START_DATE_STR,
                                                  exp_config.MAX_CASH_SESSIONS, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
candidate_period_pdf, candidate_summary_pdf = candidate_replay_dict["candidate_period_pdf"], candidate_replay_dict["candidate_summary_pdf"]
print(f"Candidates replayed in {time.time() - start_time:.0f} s")
# LOG EVERY (CANDIDATE, PERIOD) AS A VALIDATION TRIAL (ONE WRITE, FOLD COLUMNS FILLED)
validation_trial_pdf = log_trial_pdf(exp_config, "validation", candidate_period_pdf, KEY_COL_STR_LIST, [col for col in candidate_period_pdf.columns if col.startswith("valid_")],
                                     False, fold_pdf, note_str_in="exp12 step02 candidate path, one period of the continuous replay")
print(f"Validation trials logged:\t{len(validation_trial_pdf)}\tfirst logged_ts {validation_trial_pdf['logged_ts'].iloc[0]}\ttest_window_touched: {validation_trial_pdf['test_window_touched'].unique().tolist()}")
# SAVE THE CANDIDATES
write_csv_file_to_path(candidate_period_pdf, f"{output_path_str}candidate_period_data.csv", "W", alert_in=False)
write_csv_file_to_path(candidate_summary_pdf, f"{output_path_str}candidate_summary_data.csv", "W", alert_in=False)
# DISPLAY THE OPTIMISTIC BEST CANDIDATE
best_row = candidate_summary_pdf.sort_values("strategy_total_return", ascending=False).iloc[0]
print(f"Best candidate AFTER THE FACT (optimistic):\t{best_row['candidate_name']}\t{best_row['strategy_total_return']:.2%} vs buy & hold {best_row['buy_hold_total_return']:.2%}"
      f"\tlog excess {best_row['log_excess_annualized_log_excess']:.4f} [{best_row['log_excess_ci_low']:.4f}, {best_row['log_excess_ci_high']:.4f}]\tin market {best_row['in_market_pct']:.1%}\texits {int(best_row['exit_count'])}")
# PREVIEW DATAFRAME
candidate_summary_pdf[["candidate_idx", "candidate_name", "strategy_total_return", "buy_hold_total_return", "excess_total_return", "strategy_annualized_return",
                       "buy_hold_annualized_return", "in_market_pct", "exit_count", "mean_cash_sessions", "bought_back_lower_pct", "period_windows_beating_buy_hold",
                       "log_excess_annualized_log_excess", "log_excess_ci_low", "log_excess_ci_high", "strategy_max_drawdown", "buy_hold_max_drawdown"]].round(4)

In [ ]:
"""
Prior-only path (the honest validation estimate) with its baselines on the same sessions.
"""
# CALL FUNCTION TO SELECT, REPLAY THE SWITCHING PATH AND RUN THE BASELINES
start_time = time.time()
prior_dict = run_prior_only_replay_dict(ohlcv_array_dict, period_pdf, candidate_period_pdf, candidate_list, rule_builder_func, KEY_COL_STR_LIST, TIE_BREAK_LIST,
                                        exp_config.SELECTION_POOLED_QUARTER_COUNT, exp_config.UNTOUCHED_START_DATE_STR, exp_config.MAX_CASH_SESSIONS,
                                        exp_config.RANDOM_RUN_COUNT, exp_config.RANDOM_EXIT_SEED_OFFSET, exp_config.BOOTSTRAP_BLOCK_MONTH_COUNT)
summary_dict, baseline_summary_dict, log_excess_1m_dict = prior_dict["summary_dict"], prior_dict["baseline_summary_dict"], prior_dict["log_excess_1m_dict"]
print(f"Prior-only path and baselines in {time.time() - start_time:.0f} s")
# CHECK THAT THE PATH AND THE UNMODIFIED PATH COVER THE SAME SESSIONS AS THE REPRODUCED BUY-AND-HOLD
assert summary_dict["first_date"] == unmodified_summary_dict["first_date"] and summary_dict["last_date"] == unmodified_summary_dict["last_date"]
assert summary_dict["buy_hold_total_return"] == unmodified_summary_dict["buy_hold_total_return"]
# SAVE THE PATH
write_csv_file_to_path(prior_dict["selection_pdf"], f"{output_path_str}selection_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["baseline_pdf"], f"{output_path_str}prior_only_baseline_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["daily_equity_pdf"], f"{output_path_str}prior_only_daily_equity_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["simulation_dict"]["transaction_pdf"], f"{output_path_str}prior_only_transaction_data.csv", "W", alert_in=False)
write_csv_file_to_path(prior_dict["cost_sensitivity_pdf"], f"{output_path_str}prior_only_cost_sensitivity_data.csv", "W", alert_in=False)
write_csv_file_to_path(pd.DataFrame([{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}]), f"{output_path_str}prior_only_summary_data.csv", "W", alert_in=False)
# DISPLAY THE PATH AGAINST BUY AND HOLD
print(f"\nPrior-only path:\t{summary_dict['first_date']} -> {summary_dict['last_date']}\t({summary_dict['session_count']:,} sessions, {summary_dict['year_count']:.2f} years)")
print(f"Total return:\tpath {summary_dict['strategy_total_return']:.2%} | buy & hold {summary_dict['buy_hold_total_return']:.2%}\t(annualized {summary_dict['strategy_annualized_return']:.2%} vs {summary_dict['buy_hold_annualized_return']:.2%})")
print(f"Time in the market:\t{summary_dict['in_market_pct']:.1%}\tExits:\t{prior_dict['simulation_dict']['metric_dict']['signal_exit_count']}\tMean cash decisions per episode:\t{prior_dict['simulation_dict']['metric_dict']['mean_cash_sessions']:.1f}")
print(f"Periods won:\t{summary_dict['period_windows_beating_buy_hold']} of {summary_dict['period_window_count']}\t(sign test p = {summary_dict['period_sign_test_p_value']:.3f})"
      f"\tmean period excess {summary_dict['period_mean_excess_return']:.2%} (SD {summary_dict['period_std_excess_return']:.2%}, t = {summary_dict['period_t_stat']:.2f})"
      f"\tMDE {summary_dict['period_mde_per_window']:.2%} per period ({summary_dict['period_mde_per_year']:.2%} per year)")
print(f"Annualized log excess:\t{summary_dict['log_excess_annualized_log_excess']:.4f}\t95% interval (6-month blocks) [{summary_dict['log_excess_ci_low']:.4f}, {summary_dict['log_excess_ci_high']:.4f}]"
      f"\t(1-month blocks [{log_excess_1m_dict['ci_low']:.4f}, {log_excess_1m_dict['ci_high']:.4f}])\tsupported: {summary_dict['log_excess_supported']}")
print(f"Sharpe:\tpath {summary_dict['strategy_sharpe_ratio']:.2f} | buy & hold {summary_dict['buy_hold_sharpe_ratio']:.2f}\tMax drawdown:\tpath {summary_dict['strategy_max_drawdown']:.2%} | buy & hold {summary_dict['buy_hold_max_drawdown']:.2%}")
print(f"PRIMARY (beats buy & hold):\t{summary_dict['primary_success']}\tComparable:\t{summary_dict['comparable_return']}\tSecondary Sharpe / drawdown:\t{summary_dict['secondary_sharpe']} / {summary_dict['secondary_drawdown']}")
print(f"Unmodified prior-only path:\t{unmodified_summary_dict['strategy_total_return']:.2%}\tpath beats it:\t{summary_dict['strategy_total_return'] > unmodified_summary_dict['strategy_total_return']}")
print(f"Random exit (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_exit_median']:.2%}\tpath beats {baseline_summary_dict['random_exit_path_beats_share']:.0%} of runs")
print(f"Random re-entry (median of {exp_config.RANDOM_RUN_COUNT}):\t{baseline_summary_dict['random_reentry_median']:.2%}\tpath beats {baseline_summary_dict['random_reentry_path_beats_share']:.0%} of runs")
print(f"Matched probabilities:\t{prior_dict['probability_dict']}")
# APPLY THE SUCCESS AND STOPPING RULES (PROTOCOL.md §8)
path_total = summary_dict["strategy_total_return"]
information_bool = path_total > unmodified_summary_dict["strategy_total_return"] and path_total > baseline_summary_dict["random_exit_median"] and path_total > baseline_summary_dict["random_reentry_median"]
if not summary_dict["primary_success"]:
    verdict_str = "STOP"
elif information_bool:
    verdict_str = "CANDIDATE RESULT PENDING REVIEW"
else:
    verdict_str = "CANDIDATE RESULT PENDING REVIEW (information test failed: not attributable to the VIX)"
print(f"\nInformation test passed:\t{information_bool}\tVERDICT:\t{verdict_str}")
# PREVIEW THE COST SENSITIVITY
prior_dict["cost_sensitivity_pdf"].round(4)

In [ ]:
"""
Episode scorecard of the prior-only path (every exit). S / R > 1 = bought back lower than sold; reentry_trigger = M1, M2,
original rule or end of data; vix_at_exit / vix_at_reentry = vix_level_prev at the two decisions.
"""
# BUILD AND SAVE THE SCORECARD
simulation_dict = prior_dict["simulation_dict"]
scorecard_pdf = get_vix_scorecard_pdf(prior_dict["scorecard_pdf"], simulation_dict, vix_feature_pdf)
write_csv_file_to_path(scorecard_pdf, f"{output_path_str}prior_only_scorecard_data.csv", "W", alert_in=False)
# DISPLAY THE SCORECARD
if not scorecard_pdf.empty:
    print(f"Episodes:\t{len(scorecard_pdf)}\tbought back lower:\t{int((scorecard_pdf['s_over_r'] > 1).sum())}\tproduct of S/R:\t{np.prod(scorecard_pdf['s_over_r']):.4f}"
          f"\tmean sessions out:\t{scorecard_pdf['sessions_out'].mean():.1f}\ttotal cost:\t{scorecard_pdf['cost_paid_pct'].sum():.4%} of the sale values")
    print("Re-entry reasons:", scorecard_pdf["reentry_trigger"].value_counts().to_dict())
scorecard_pdf[["exit_date", "exit_fill_price", "reentry_date", "reentry_fill_price", "s_over_r", "sessions_out", "reentry_trigger", "vix_at_exit", "vix_at_reentry",
               "buy_hold_return_out", "max_decline_out_pct", "cost_paid_pct", "open_at_end"]].round(4) if not scorecard_pdf.empty else scorecard_pdf

In [ ]:
"""
Pre-registered concentration reports (no gate): (a) final equity / buy-and-hold = product of S / R, without the largest
one and two episodes; (b) the log excess per calendar year; (c) the share of the total log excess from the episodes that
overlap 2020-02-01 -> 2020-06-30.
"""
# CALCULATE THE REPORTS
path_equity_pdf, buy_hold_equity_pdf = simulation_dict["daily_equity_pdf"], prior_dict["buy_hold_dict"]["daily_equity_pdf"]
concentration_dict = get_concentration_dict(scorecard_pdf, path_equity_pdf, buy_hold_equity_pdf)
year_pdf = get_yearly_log_excess_pdf(path_equity_pdf, buy_hold_equity_pdf)
# SAVE THE REPORTS
write_csv_file_to_path(pd.DataFrame([concentration_dict]), f"{output_path_str}concentration_data.csv", "W", alert_in=False)
write_csv_file_to_path(year_pdf, f"{output_path_str}yearly_log_excess_data.csv", "W", alert_in=False)
# DISPLAY (a)
print(f"(a) final equity / buy & hold:\t{concentration_dict['equity_ratio']:.4f}\tproduct of S/R:\t{concentration_dict['product_s_over_r']:.4f}\t({concentration_dict['episode_count']} episodes)")
print(f"    without the largest log(S/R):\tequity ratio {concentration_dict['equity_ratio_without_top1']:.4f}\tproduct {concentration_dict['product_s_over_r_without_top1']:.4f}\tbeats buy & hold: {concentration_dict['beats_buy_hold_without_top1']}")
print(f"    without the two largest:\tequity ratio {concentration_dict['equity_ratio_without_top2']:.4f}\tproduct {concentration_dict['product_s_over_r_without_top2']:.4f}\tbeats buy & hold: {concentration_dict['beats_buy_hold_without_top2']}")
# DISPLAY (c)
print(f"(c) episodes overlapping {exp_config.CONCENTRATION_WINDOW_START_STR} -> {exp_config.CONCENTRATION_WINDOW_END_STR}:\t{concentration_dict['window_episode_count']}"
      f"\tsum of log(S/R) {concentration_dict['window_log_share_gain']:.4f}\ttotal log excess {concentration_dict['total_log_excess']:.4f}\tshare {concentration_dict['window_share_of_log_excess']:.2f}")
# DISPLAY (b)
print("(b) log excess per calendar year:")
year_pdf.round(4)

In [ ]:
# DISPLAY THE SELECTED CANDIDATE PER PERIOD (THE CHOICE OF PERIOD f GOVERNS PERIOD f + 1)
selection_pdf = prior_dict["selection_pdf"]
print("Selections (candidate: periods chosen):", selection_pdf.groupby("candidate_name").size().to_dict())
governed_name_list = [candidate_list[prior_dict["period_candidate_dict"][period_id]]["candidate_name"] for period_id in sorted(prior_dict["period_candidate_dict"])]
print("Periods governed by each candidate:", pd.Series(governed_name_list).value_counts().to_dict())
selection_pdf[["fold_id", "candidate_name", "pooled_score", "pooled_quarter_count"]].round(4)

In [ ]:
# CREATE A FIGURE OF THE PRIOR-ONLY PATH vs THE UNMODIFIED PATH vs BUY AND HOLD (GROWTH OF 1)
fig = go.Figure()
fig.add_trace(go.Scatter(x=path_equity_pdf["date"], y=path_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="prior-only path (exp12)", line={"width": 1.5}))
fig.add_trace(go.Scatter(x=unmodified_prior_dict["simulation_dict"]["daily_equity_pdf"]["date"], y=unmodified_prior_dict["simulation_dict"]["daily_equity_pdf"]["equity"] / config.INITIAL_CAPITAL,
                         name="unmodified prior-only path {E1, E2}", line={"width": 1.0}))
fig.add_trace(go.Scatter(x=buy_hold_equity_pdf["date"], y=buy_hold_equity_pdf["equity"] / config.INITIAL_CAPITAL, name="buy & hold", line={"width": 1.5}))
fig.update_layout(template="plotly_dark", width=1100, height=450, title="exp12 prior-only path vs unmodified exits vs buy-and-hold (continuous replay, growth of 1)")
fig.show()

In [ ]:
# LOG ONE SUMMARY ENTRY OF THE RUN (THE CANDIDATE PERIODS ARE ALREADY LOGGED; THIS RECORDS THE HONEST ESTIMATE THAT WAS SEEN)
summary_trial_dict = log_trial_dict(exp_config, "summary", {"run_mode": RUN_MODE_STR, "selection": f"pooled {exp_config.SELECTION_POOLED_QUARTER_COUNT} periods", "path": "prior-only continuous replay"},
                                    {**{key: value for key, value in summary_dict.items() if key != "baselines_beaten"}, **baseline_summary_dict,
                                     "log_excess_1m_ci_low": log_excess_1m_dict["ci_low"], "log_excess_1m_ci_high": log_excess_1m_dict["ci_high"],
                                     "exit_count": simulation_dict["metric_dict"]["signal_exit_count"], "verdict": verdict_str, "information_test": information_bool,
                                     "unmodified_prior_only_total_return": unmodified_summary_dict["strategy_total_return"], **reproduction_dict,
                                     **{f"concentration_{key}": value for key, value in concentration_dict.items()},
                                     "best_after_the_fact_total_return": float(candidate_summary_pdf["strategy_total_return"].max())},
                                    False, note_str_in="exp12 step02 validation summary (prior-only continuous path, unmodified path, baselines, concentration)")
print(f"Summary logged:\t{summary_trial_dict['trial_id']}\ttest_window_touched: {summary_trial_dict['test_window_touched']}")

In [ ]:
# CALL FUNCTION TO READ THE TRIAL LOG
trial_log_pdf = read_trial_log_pdf()
# DISPLAY THE NUMBER OF TRIALS PER EXPERIMENT AND STAGE (MULTIPLE-TESTING RECORD)
get_trial_count_pdf(trial_log_pdf)